In [1]:
from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import Chroma
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from dotenv import load_dotenv

load_dotenv()

c:\Users\-\OneDrive\Desktop\FYP2_P1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


True

In [2]:
# Robust file loader: try common encodings and fall back to a safe decode
def load_text_file(path):
    for enc in ("utf-8", "utf-8-sig", "latin-1", "cp1252"):
        try:
            with open(path, encoding=enc) as f:
                return f.read()
        except Exception:
            continue
    with open(path, "rb") as f:
        return f.read().decode("utf-8", errors="replace")

text = load_text_file("script.txt")
# Create a single Document from the loaded text (langchain core Document already imported)
docs = [Document(page_content=text)]
batches=[batch.strip() for batch in docs[0].page_content.split('\n') if batch.strip()]

In [3]:
#schema for semantic chunking
json_schema = {
    "title": "SemanticChunks",
    "description": "Semantic chunking of document into meaningful segments",
    "type": "object",
    "properties": {
        "chunks": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "category": {
                        "type": "string",
                        "description": "Main topic category of this chunk"
                    },
                    "content": {
                        "type": "string",
                        "description": "The text content of this semantic chunk"
                    },
                    "summary": {
                        "type": "string",
                        "description": "Brief summary of this chunk"
                    }
                },
                "required": ["category", "content", "summary"]
            }
        }
    },
    "required": ["chunks"]
}

In [4]:
import json
# Process each batch
results = []
for i, batch in enumerate(batches):
    print(f"Processing batch {i+1}/{len(batches)}...")

    model = ChatOpenAI(model="gpt-4o-mini")
    structured_model = model.with_structured_output(schema=json_schema)

    template = PromptTemplate(
        template="""Please do semantic chunking of the following data {data}, please do not miss any information, and make sure to categorize the chunks appropriately.""",
        input_variables=["data"]
    )

    chain = template | structured_model
    result = chain.invoke({"data": batch})
    results.append(result)
    print(f"Batch {i+1} completed")

# Combine all results
all_chunks = []
for result in results:
    if 'chunks' in result:
        all_chunks.extend(result['chunks'])

final_result = {"chunks": all_chunks}
print("\nFinal Result:", json.dumps(final_result, indent=2, ensure_ascii=False))

Processing batch 1/18...
Batch 1 completed
Processing batch 2/18...
Batch 2 completed
Processing batch 3/18...
Batch 3 completed
Processing batch 4/18...
Batch 4 completed
Processing batch 5/18...
Batch 5 completed
Processing batch 6/18...
Batch 6 completed
Processing batch 7/18...
Batch 7 completed
Processing batch 8/18...
Batch 8 completed
Processing batch 9/18...
Batch 9 completed
Processing batch 10/18...
Batch 10 completed
Processing batch 11/18...
Batch 11 completed
Processing batch 12/18...
Batch 12 completed
Processing batch 13/18...
Batch 13 completed
Processing batch 14/18...
Batch 14 completed
Processing batch 15/18...
Batch 15 completed
Processing batch 16/18...
Batch 16 completed
Processing batch 17/18...
Batch 17 completed
Processing batch 18/18...
Batch 18 completed

Final Result: {
  "chunks": [
    {
      "category": "Attendance Requirements",
      "content": "FAST-NUCES ke students se 80% attendance undergraduate programs mein aur 75% graduate aur postgraduate progr

In [5]:
len(all_chunks)

247

In [6]:
Documents=[Document(page_content=chunk["content"], metadata={"category": chunk["category"]}) for chunk in all_chunks]

In [7]:
vector_store = Chroma(
    embedding_function=OpenAIEmbeddings(),
    persist_directory='chroma_db',
    collection_name='sample'
)

C:\Users\-\AppData\Local\Temp\ipykernel_8220\318618104.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_store = Chroma(


In [8]:
vector_store.add_documents(Documents)

['c760be12-c45b-4434-a962-d7650e09d391',
 '6a5ae145-8b1a-48bf-860b-09d17af5b07c',
 '34926599-b968-4eb1-b91c-062542f11b1d',
 '2fadebef-b77e-42b4-932c-fb235291e015',
 'd91780a6-5b7b-44c3-b596-ccf809ece855',
 'd35ea4b3-e1f7-4034-9440-4e08c86ff87b',
 '3eca0c82-323e-42ad-b1d2-18423dd60326',
 'fac565d2-62b7-47d9-af25-e5732db29302',
 'ee1eebaa-4883-4960-9644-9d6bd7de3b67',
 '356e7c04-a578-4993-923e-248f4fa1a411',
 'c178ce29-e4c6-4eaf-949a-b99a2a99e59d',
 'c8121823-7deb-4d08-9159-05cf721ec7f9',
 '27f23fdc-bd72-477e-8d31-12a9d1048f12',
 '2d6ff437-d194-47c7-b9a9-73c6aa14ed99',
 'd578f56a-327e-4314-adee-b7bb39505d06',
 'd4f6826a-271d-49ee-bdb9-dfbdf7b34ca8',
 '0c528659-ee49-4274-b0fc-1d84775dafa3',
 '61974c8b-c4fd-4402-88a9-d591ed0a38c9',
 '1f734250-96b3-402e-95f4-9a6746814405',
 '12b3293b-6dbc-445e-959c-b2f4d6fd7b1c',
 '1f8e637c-6a26-4ad7-9b42-6b538014087a',
 'd3ef2976-19b3-426d-930e-83af68788f57',
 'aec54686-3d3e-484f-b539-f7141e6d479d',
 '078aaa47-3cbf-4e31-aaf7-7089b3ddd73c',
 '2e334049-89ab-

In [9]:
for chunk in all_chunks:
    print(chunk)

{'category': 'Attendance Requirements', 'content': 'FAST-NUCES ke students se 80% attendance undergraduate programs mein aur 75% graduate aur postgraduate programs mein talab ki jati hai; jismein maximum 20% absence haqeeqi wajah jaise bemari ke liye maaf ki jati hai. Agar yeh requirement poori na ki gayi to wo final exams mein baithne ke laayak nahi rahte aur course mein F/A grade milta hai.', 'summary': 'Attendance requirements for students, including allowable absences.'}
{'category': 'Language of Instruction', 'content': 'Taleemi zaban aur imtihan ki zaban English hai, siwaye language courses aur Religious and Islamic Studies ke jahan students English ya Urdu mein jawab de sakte hain.', 'summary': 'Medium of instruction and examination is primarily English, with some exceptions.'}
{'category': 'Academic Calendar', 'content': 'Academic saal August/September se May/June tak hota hai jis mein Fall aur Spring semesters hote hain; har semester takriban 16 haftay plus makeup aur imtihan 

In [10]:
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['3a5cace3-c972-4e2d-9430-0374f99cbd1c',
  '9dca5b18-d24d-4747-905b-f634d02aed39',
  '91c9da65-ce20-4821-b672-ecaff24d0285',
  'cbd273e6-d4cb-4088-bad2-1fd537c92aa5',
  '5ced54ae-5c08-4959-8e59-4634855dcd0c',
  '7a44b3a8-5d8f-4616-9c9c-8408bf53c157',
  '60da3c94-d22c-4fff-b71b-d5c9517aea09',
  '3ffc7f6e-cdc4-4563-a919-c96b97640fad',
  '7b741d91-f8a6-4b5c-b32f-a994da56d44b',
  'd7a4b49b-ac20-4bb8-a766-b18d57a485c4',
  '31a41492-a2a3-492c-90bc-d8de79cb5814',
  'aa9b9aaa-302c-4913-b47b-7cb60e0e1557',
  '8a31f9eb-51cf-4d8d-8c70-5440a626913f',
  'b1b9fd3d-b2af-41f6-85d9-a9c3a4408fcc',
  '100d42fd-4b8d-4f54-8fab-d8401130d6d3',
  'a451ab3d-7a68-4195-9f28-7067dda9eb64',
  'efad68d9-85ff-4fb2-a793-2fdc73313eb5',
  '461e63ad-821c-4b17-8871-5873cc7f1675',
  '82acd2b6-27f5-4e58-99df-661f5ebbf86a',
  '7351ce2e-af35-4c32-8704-95abbeea5d85',
  'f067f9e4-7cb4-4fa0-9e24-162081f8458b',
  '293639b2-c7a5-4c28-aada-47fc4b3a47bd',
  '28f2c370-464e-44d1-adab-95519d9720da',
  '4c8ec32c-2cbe-4c8b-93dc-

In [11]:
query='fast kee tuition fee kitne hain?'

In [12]:
vector_store.similarity_search(query=query,k=15)

[Document(metadata={'category': 'Fee Payment'}, page_content='Fees ki payment admission par mukammal hoti hai jo university prospectus mein tafseeli hai.'),
 Document(metadata={'category': 'Fee Structure'}, page_content='Fees ki payment admission par mukammal hoti hai jo university prospectus mein tafseeli hai.'),
 Document(metadata={'category': 'FAST Fellowship'}, page_content='FAST Fellowship ke tehat select kiye gaye PhD students ko Rs 60,000 per month stipend, full tuition fee waiver, aur equipment aur travel expenses ke liye alag fund diya jata hai.'),
 Document(metadata={'category': 'Tuition Fees'}, page_content="Tuition Fee har semester start hone se pehle mukammal ada karna hoti hai aur saal bhar mein dobara review hoti hai; academic saal 2025-26 mein Master aur PhD programs ke sath saath sabhi campuses aur programs ke Bachelor's degrees ke liye tuition fee Rs. 11,000 per credit hour hai."),
 Document(metadata={'category': 'Tuition Fee'}, page_content="Tuition Fee har semester 

In [13]:
# RAG: Retrieval Augmentation + Generation with PromptTemplate and GPT-4o-mini
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

retrieved_docs = retriever.invoke(query)

context = "\n\n".join(
    [f"[{i+1}] Category: {doc.metadata.get('category', 'N/A')}\n{doc.page_content}" for i, doc in enumerate(retrieved_docs)]
)

rag_prompt = PromptTemplate(
    template="""
You are a helpful assistant for FAST-NUCES queries.
Answer the user using ONLY the provided context.
If the answer is not present in the context, clearly say you don't have enough information.
Keep the answer concise and accurate.

Question: {question}

Context:
{context}

Answer:
""",
    input_variables=["question", "context"]
)

# Define JSON schema for structured output with Roman Urdu and Arabic Urdu script
output_schema = {
    "title": "BilingualResponse",
    "description": "Response in both Roman Urdu and Arabic Urdu script",
    "type": "object",
    "properties": {
        "roman_urdu": {
            "type": "string",
            "description": "Response text in Roman Urdu (English script)"
        },
        "arabic_urdu": {
            "type": "string",
            "description": "Response text in Arabic Urdu script"
        }
    },
    "required": ["roman_urdu", "arabic_urdu"]
}

model = ChatOpenAI(model="gpt-4o-mini", temperature=0.1)
structured_model = model.with_structured_output(schema=output_schema)
final_prompt = rag_prompt.format(question=query, context=context)
response = structured_model.invoke(final_prompt)

print("Answer:\n", response["roman_urdu"], response["arabic_urdu"],sep="\n")

Answer:

FAST ki tuition fee Rs. 11,000 per credit hour hai.
فاسٹ کی ٹیوشن فیس Rs. 11,000 فی کریڈٹ گھنٹہ ہے۔
